# Multi-armed bandits: a midpoint change and timing sensitivity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/reinforcement-learning/multi_armed_bandits.ipynb)

All data are generated locally. This notebook runs on CPU without credentials or repository files.

Dependencies: numpy, scipy, matplotlib, pillow. If needed, install these packages with `%pip install` before running the imports.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

# Keep generated figures in a temporary folder for this notebook session.
figure_folder = TemporaryDirectory()
IMG = Path(figure_folder.name)
COLOURS = ['#167d8d', '#e6a23b', '#526b98']
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 120})

def show_table(columns, rows):
    header = '| ' + ' | '.join(columns) + ' |'
    separator = '| ' + ' | '.join(['---'] * len(columns)) + ' |'
    body = ['| ' + ' | '.join(map(str, row)) + ' |' for row in rows]
    display(Markdown('\n'.join([header, separator] + body)))

In [ ]:
from scipy.stats import beta
from matplotlib.animation import FuncAnimation, PillowWriter

## Rewards, actions and posterior counts

Five arms have Bernoulli probabilities 0.1 through 0.5. A policy observes only its chosen reward. Each independent repetition retains its own successes and counts. The optional `change_at` counts completed rounds before probabilities reverse. Thus 1000 means the new probabilities first apply at round 1001.

Thompson sampling uses Beta(1+successes, 1+counts−successes). The code checks its analytical posterior counts whenever a snapshot is recorded. UCB1 and epsilon-greedy use empirical means after each arm has been sampled once. Exact argmax ties favour the lowest arm index in this implementation.

In [ ]:
# Run one policy with shared per-round potential rewards and selected-arm feedback.
def bandit_run(method, drift=False, repeats=100, horizon=2000, seed=17, change_at=None):
    env = np.random.default_rng(seed)
    rng = np.random.default_rng(seed + 100)
    probabilities = np.array([0.1, 0.2, 0.3, 0.4, 0.5])
    streams = env.random((horizon, repeats, 5))
    rows = np.arange(repeats)
    count = np.zeros((repeats, 5))
    success = np.zeros_like(count)
    regret = np.zeros((repeats, horizon))
    chosen = np.zeros((horizon, 5))
    snapshots = {}
    if repeats < 2 or horizon < 5:
        raise ValueError('Use at least two repetitions and five rounds.')
    if change_at is None and drift:
        change_at = horizon // 2
    if change_at is not None and (not 0 < change_at < horizon):
        raise ValueError('change_at counts completed rounds before the change.')
    snapshot_rounds = set(np.unique(np.geomspace(5, horizon, 11).astype(int))) | {horizon}
    for t in range(horizon):
        p = probabilities[::-1] if change_at is not None and t >= change_at else probabilities
        if t < 5:
            arm = np.full(repeats, t)
        elif method == 'Thompson':
            arm = rng.beta(1 + success, 1 + count - success).argmax(axis=1)
        elif method == 'UCB1':
            arm = (success / count + np.sqrt(2 * np.log(t + 1) / count)).argmax(axis=1)
        elif method == 'Epsilon 0.1':
            arm = (success / count).argmax(axis=1)
            explore = rng.random(repeats) < 0.1
            arm[explore] = rng.integers(0, 5, size=explore.sum())
        else:
            raise ValueError(method)
        reward = (streams[t, rows, arm] < p[arm]).astype(float)
        count[rows, arm] += 1
        success[rows, arm] += reward
        regret[:, t] = p.max() - p[arm]
        chosen[t] = np.bincount(arm, minlength=5) / repeats
        if t + 1 in snapshot_rounds:
            posterior_alpha = 1 + success[0].copy()
            posterior_beta = 1 + count[0] - success[0]
            assert np.allclose(posterior_alpha - 1, success[0])
            assert np.allclose(posterior_beta - 1, count[0] - success[0])
            assert np.allclose(posterior_alpha + posterior_beta, count[0] + 2)
            snapshots[t + 1] = (posterior_alpha, posterior_beta)
    return {'regret': np.cumsum(regret, axis=1), 'chosen': chosen, 'snapshots': snapshots, 'count': count, 'success': success, 'change_at': change_at, 'horizon': horizon, 'repeats': repeats}

## Repeated comparisons and standard errors

The stationary and midpoint-change comparisons each use 100 repetitions of 2000 rounds. Pseudo-regret compares each action with the best arm at that time. Standard errors divide the across-repetition standard deviation by sqrt(repetitions), derived from the returned array rather than hard-coded at 100 runs.

In [ ]:
# Summarise final regret and verify each repetition used the requested horizon.
def bandit_experiment(repeats=100, horizon=2000):
    runs = {}
    summary = {}
    for scenario in ['stationary', 'changed']:
        runs[scenario] = {}
        summary[scenario] = {}
        for name in ['Thompson', 'UCB1', 'Epsilon 0.1']:
            run = bandit_run(name, scenario == 'changed', repeats=repeats, horizon=horizon)
            runs[scenario][name] = run
            final = run['regret'][:, -1]
            summary[scenario][name] = {'mean_regret': float(final.mean()), 'se': float(final.std(ddof=1) / np.sqrt(len(final))), 'median_regret': float(np.median(final))}
            assert np.all(run['count'].sum(axis=1) == horizon)
            assert np.all(run['success'] <= run['count'])
    return (summary, runs)

In [ ]:
summary, runs = bandit_experiment(repeats=100, horizon=2000)
rows = []
for scenario, methods in summary.items():
    for method, result in methods.items():
        rows.append([scenario, method, f"{result['mean_regret']:.2f}",
                     f"{result['se']:.2f}"])
show_table(['Environment', 'Policy', 'Mean regret', 'Standard error'], rows)

# Verify the exercise parameters on a smaller independent check.
small_summary, small_runs = bandit_experiment(repeats=7, horizon=40)
for scenario, methods in small_runs.items():
    for method, run in methods.items():
        assert run['regret'].shape == (7, 40)
        assert np.all(run['count'].sum(axis=1) == 40)
        expected_se = run['regret'][:, -1].std(ddof=1) / np.sqrt(7)
        assert np.isclose(small_summary[scenario][method]['se'], expected_se)

## Change timing sensitivity

A midpoint change does not establish a general ranking for changing environments. The next function holds the horizon, arm probabilities and seeds fixed while changing only the number of completed rounds before reversal. All results compare mean total regret over the entire horizon.

In [ ]:
# Check changes after 500, 1000 and 1800 completed rounds.
def bandit_sensitivity(repeats=100, horizon=2000):
    rows = []
    for change_at in [500, 1000, 1800]:
        row = {'change_after_round': change_at}
        for method in ['Thompson', 'UCB1']:
            run = bandit_run(method, repeats=repeats, horizon=horizon, change_at=change_at)
            final = run['regret'][:, -1]
            row[method] = {'mean_regret': float(final.mean()), 'se': float(final.std(ddof=1) / np.sqrt(repeats))}
        rows.append(row)
    return rows

In [ ]:
sensitivity = bandit_sensitivity()
show_table(['Change after completed round', 'Thompson mean regret', 'UCB1 mean regret'],
    [[r['change_after_round'], f"{r['Thompson']['mean_regret']:.2f}",
      f"{r['UCB1']['mean_regret']:.2f}"] for r in sensitivity])
paired_rows = []
for scenario, methods in runs.items():
    difference = methods['Thompson']['regret'][:, -1] - methods['UCB1']['regret'][:, -1]
    paired_rows.append([scenario, f'{difference.mean():.2f}',
                       f'{difference.std(ddof=1)/np.sqrt(len(difference)):.2f}'])
show_table(['Environment', 'Paired Thompson minus UCB1', 'Standard error'], paired_rows)

## Regret, allocations and posterior evolution

The regret bands describe uncertainty in the mean across repetitions. The allocation chart averages action choices in 50-round bins. The posterior GIF follows the first stationary repetition only; its changing y-axis makes early densities readable, and dotted lines mark true probabilities. A final hold gives time to read the learned distributions.

In [ ]:
# Animate posteriors with true-probability markers and adaptive density limits.
def bandit_posteriors(run):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    grid = np.linspace(0.001, 0.999, 400)
    snapshots = run['snapshots']
    frames = list(snapshots) + [list(snapshots)[-1]] * 5
    colours = plt.get_cmap('tab10').colors

    def update(round_number):
        ax.clear()
        alpha, beta_parameter = snapshots[round_number]
        peak = 0
        for arm, probability in enumerate([0.1, 0.2, 0.3, 0.4, 0.5]):
            density = beta.pdf(grid, alpha[arm], beta_parameter[arm])
            peak = max(peak, float(density.max()))
            ax.plot(grid, density, color=colours[arm], label=f'Arm {arm} (p={probability:.1f})')
            ax.axvline(probability, color=colours[arm], ls=':', alpha=0.6, lw=1)
        ax.set(title=f'Thompson posterior: one repetition after {round_number} rounds', xlabel='Reward probability (dotted lines mark truth)', ylabel='Posterior density', ylim=(0, peak * 1.2))
        ax.legend(loc='upper left', ncol=3, fontsize=9)
        fig.tight_layout()
    return (FuncAnimation(fig, update, frames=frames, interval=400, repeat=True), fig)

In [ ]:
# Generate all three article figures using the actual run dimensions.
def bandit_figures(summary, runs):
    slug = 'multi-armed-bandit-python'
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    for ax, (scenario, methods) in zip(axes, runs.items()):
        for colour, (name, run) in zip(COLOURS, methods.items()):
            regret = run['regret']
            mean = regret.mean(axis=0)
            se = regret.std(axis=0, ddof=1) / np.sqrt(regret.shape[0])
            rounds = np.arange(1, regret.shape[1] + 1)
            ax.plot(rounds, mean, label=name, color=colour)
            ax.fill_between(rounds, mean - 1.96 * se, mean + 1.96 * se, alpha=0.15, color=colour)
        if run['change_at'] is not None:
            ax.axvline(run['change_at'], color='grey', ls='--')
        ax.set(title=scenario.capitalize(), xlabel='Round', ylabel='Cumulative pseudo-regret')
    axes[0].legend(fontsize=9)
    fig.tight_layout()
    fig.savefig(IMG / f'{slug}-regret.webp')
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    selection = runs['stationary']['Thompson']['chosen']
    starts = list(range(0, len(selection), 50))
    smooth = np.array([selection[i:i + 50].mean(axis=0) for i in starts])
    centres = [i + (len(selection[i:i + 50]) - 1) / 2 + 1 for i in starts]
    ax.stackplot(centres, smooth.T, labels=[f'p={p:.1f}' for p in [0.1, 0.2, 0.3, 0.4, 0.5]], alpha=0.85)
    ax.set(xlabel='Round (50-round bins)', ylabel='Fraction of selections', ylim=(0, 1))
    ax.legend(ncol=5, fontsize=9)
    fig.tight_layout()
    fig.savefig(IMG / f'{slug}-allocation.webp')
    plt.close(fig)
    animation, fig = bandit_posteriors(runs['stationary']['Thompson'])
    animation.save(IMG / f'{slug}-posteriors.gif', writer=PillowWriter(fps=3))
    plt.close(fig)

In [ ]:
bandit_figures(summary, runs)
for name in ['regret.webp', 'allocation.webp', 'posteriors.gif']:
    display(Image(filename=str(IMG / f'multi-armed-bandit-python-{name}')))

## Exercises

1. Change the Beta prior to Beta(2,2) and update the analytical-count assertions consistently.
2. Compare epsilon values using a separate evaluation stream after selection.
3. Add a windowed or discounted policy for a stated change model.
4. Run `bandit_experiment(repeats=1000)` and compare Monte Carlo standard errors. The division uses sqrt(1000), not the default sqrt(100).

Thompson sampling wins the stationary instance. UCB1 wins the change-after-1000 instance, while Thompson wins when the change is delayed until after 1800 rounds. These are finite-horizon instance comparisons.

References: [UCB1](https://homes.di.unimi.it/~cesabian/Pubblicazioni/ml-02.pdf), [Thompson sampling](https://arxiv.org/abs/1707.02038).